In [76]:
import sys
from torch.profiler import profile, ProfilerActivity, record_function
import tensorly as tl
import itertools
from torch.utils.data import DataLoader
import torchinfo

TDECOMP_PATH = '..'
if not TDECOMP_PATH in sys.path:
    sys.path.append(TDECOMP_PATH)

import mlflow
import os

mlflow.set_tracking_uri("http://172.19.0.1:5000")
mlflow.set_experiment("my-first-experiment-glazkov")
mlFlowClient = mlflow.MlflowClient()
# client = mlflow.server.get_app_client("basic-auth", "http://172.19.0.1:5000")
# client.create_user(username="minio", password="minio123")
# # mlflow.export

os.environ["AWS_ACCESS_KEY_ID"] = "minio"
os.environ["AWS_SECRET_ACCESS_KEY"] = "minio123"
os.environ["MLFLOW_S3_ENDPOINT_URL"] = f"http://172.19.0.1:9000" #https://github.com/mlflow/mlflow/issues/2150


In [3]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
import tdecomp.matrix.functional as F 
from tdecomp.grad_proj.tensorgrad.prepared_tg import ParallelTG, ULTG


MODEL_NAME = "arnir0/Tiny-LLM"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME,  use_fast=False)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)


2026-02-26 08:38:28.577863: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-02-26 08:38:28.797313: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-02-26 08:38:30.133895: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
Loading weights: 100%|██████████| 12/12 [00:00<00:00, 800.90it/s, Materializing 

In [4]:
# train_imdb.py
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
    pipeline
)
from datasets import load_dataset
import torch
# from peft import LoraConfig, get_peft_model, TaskType
import evaluate
import numpy as np

# model_name = "Qwen/Qwen2-0.5B-Instruct"
model_name = 'arnir0/Tiny-LLM'
# output_dir = "./qwen2-0.5b-imdb-finetuned"
output_dir = './tiny-llm'
max_length = 512  # Maximum context length for each sample



print("Loading model and tokenizer...")

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True, use_fast=True)
# Set padding token if it doesn't exist
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Load model with 4-bit quantization if enabled
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="cuda:0", #auto
    torch_dtype=torch.float32,
    trust_remote_code=True
)

dataset = load_dataset("wikitext", "wikitext-2-raw-v1", split="train[:1000]")

print(f"Dataset size: {len(dataset)}")
print(f"Dataset features: {dataset.features}")

# Preprocess the dataset based on its structure
def preprocess_dataset(examples):
    """Extract text from different dataset formats"""
    if 'text' in examples:
        return {"text": examples["text"]}
    elif 'article' in examples:  # CNN Daily Mail
        return {"text": examples["article"]}
    elif 'content' in examples:  # Some datasets
        return {"text": examples["content"]}
    elif 'sentence' in examples:  # Some sentence datasets
        return {"text": examples["sentence"]}
    else:
        # Try to use the first string column
        for key, value in examples.items():
            if isinstance(value[0], str):
                return {"text": examples[key]}
        return {"text": [str(x) for x in examples[list(examples.keys())[0]]]}

# Apply preprocessing
dataset = dataset.map(preprocess_dataset, batched=True)

# Filter out empty texts
dataset = dataset.filter(lambda example: len(example["text"].strip()) > 0)

MAX_CONTEXT_LENTGH = 512
# Tokenization function
def tokenize_function(examples):
    tokenized = tokenizer(
        examples["text"],
        truncation=True,
        padding=True,
        max_length=MAX_CONTEXT_LENTGH, 
        return_tensors="pt"
    )
    tokenized["labels"] = tokenized["input_ids"].clone()
    return tokenized

# Tokenize the dataset
tokenized_dataset = dataset.map(tokenize_function, batched=True)

# Split dataset
train_test_split = tokenized_dataset.train_test_split(test_size=0.2, seed=42)
train_dataset = train_test_split["train"]
eval_dataset = train_test_split["test"]

print(f"Training samples: {len(train_dataset)}")
print(f"Validation samples: {len(eval_dataset)}")


# This will dynamically pad the batches during training
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,  # We are doing causal LM, not masked LM
)


print("MODEL DEVICE", model.device)
print("DATASET DEVICE (CPU)", next(iter(train_dataset))['input_ids']) 
print(len(next(iter(train_dataset))['input_ids']))

[INFO] Running in WANDB offline mode
Loading model and tokenizer...


`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 12/12 [00:00<00:00, 497.86it/s, Materializing param=model.norm.weight]                            


Dataset size: 1000
Dataset features: {'text': Value('string')}
Training samples: 517
Validation samples: 130
MODEL DEVICE cuda:0
DATASET DEVICE (CPU) [1, 29871, 353, 353, 353, 2921, 1997, 21043, 353, 353, 353, 29871, 13, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,

In [5]:
# ----------------------------
# 7. Training Arguments
# ----------------------------
# Training arguments
num_train_epochs = 3
per_device_train_batch_size = 4
per_device_eval_batch_size = 4
gradient_accumulation_steps = 1
learning_rate = 2e-4
logging_steps = 5

training_args = TrainingArguments(
    output_dir=output_dir,
    num_train_epochs=num_train_epochs,
    per_device_train_batch_size=per_device_train_batch_size,
    per_device_eval_batch_size=per_device_eval_batch_size,
    gradient_accumulation_steps=gradient_accumulation_steps,
    learning_rate=learning_rate,
    # logging_steps=logging_steps,
    logging_steps=logging_steps, #force model to make logs and WITHOUT eval (see table with each 5 step logs)
    save_steps=500,
    eval_strategy="no",
    #eval_steps=5, #every evaluate() in trainer consume lots cpu at the end to copy all metrics to cpu
    save_total_limit=2,
    load_best_model_at_end=True,
    eval_accumulation_steps=None, #if not batch eval, will send every eval_acum_steps [loses, preds, labels, inputs] on cpu inside epoch
    save_strategy="no", #'steps' by default
    #batch_eval_metrics=True, #will eval metrics every train batch! False - will eval metrics in the end of train epoch or in eval_steps=5
    report_to="mlflow",  # Disable external logging like Weights & Biases for simplicity
    fp16=False,  # Use mixed precision training #TODO try bf16
    bf16=True
)


In [6]:
def get_batch_size_mb(batch_encoding):
    """Получить размер BatchEncoding в МБ (суммирует все тензоры)"""
    total_bytes = sum(
        v.numel() * v.element_size() 
        for v in batch_encoding.values() 
        if torch.is_tensor(v)
    )
    return total_bytes / (1024 ** 2)

def get_model_size_mb(model):
    """Подсчитать примерный размер модели в МБ"""
    total_params = 0
    total_bytes = 0
    
    for param in model.parameters():
        num_params = param.numel()
        param_bytes = num_params * param.element_size()
        total_params += num_params
        total_bytes += param_bytes
    
    size_mb = total_bytes / (1024 ** 2)
    
    return size_mb

In [7]:
def create_optimizer(rank, sparse_ratio, lambda_sparse, update_gap):
    parallel_tg_optimizer = ParallelTG(model, 
                        tl.truncated_svd, 
                        (rank, sparse_ratio), #rank and ratio (but firstly will be sparce projection)
                        n_train=len(train_dataset),
                        batch_size=per_device_train_batch_size,
                        # scheduler='StepLR',
                        tensorgrad_sum_lambda_sparse=lambda_sparse,
                        update_proj_gap=update_gap
                        )
    return parallel_tg_optimizer
parallel_tg_optimizer = create_optimizer(150, 0.05, 0.05, 20)
parallel_tg_optimizer

(TensorGRaD (
 Parameter Group 0
     betas: (0.9, 0.999)
     correct_bias: True
     eps: 1e-06
     initial_lr: 0.0001
     lr: 0.0001
     weight_decay: 0.0
 
 Parameter Group 1
     batch_size: 4
     betas: (0.9, 0.999)
     correct_bias: True
     dim: 2
     enforce_full_complex_precision: False
     epochs: 100
     eps: 1e-06
     galore_2d_proj_type: left
     initial_lr: 0.0001
     lambda_sparse: 0.05
     lr: 0.0001
     n_iter_max_tucker: 10
     optimizer_type: tensorgrad_sum
     proj_type: low_rank
     rank: 150
     reset_sparse_optimizer_states: False
     scale: 1.0
     scale_by_mask_ratio: True
     scheduler_T_max: 100
     second_proj_type: unstructured_sparse
     second_rank: 128
     second_scale: 1.0
     second_scale_by_mask_ratio: False
     second_sparse_ratio: 0.25
     second_sparse_type: topk
     sparse_ratio: 0.05
     sparse_type: topk
     svd_type: <function truncated_svd at 0x7e47c5e2e980>
     training_samples: 517
     tucker_warm_restart: Tr

In [8]:
parallel_tg_optimizer[0].param_groups[1]

{'params': [Parameter containing:
  tensor([[ 5.1498e-04,  8.9216e-04, -9.9945e-04,  ..., -4.0936e-04,
            7.7057e-04, -1.8244e-03],
          [-1.3294e-03, -1.7366e-03, -9.8705e-04,  ...,  3.0041e-04,
            1.3943e-03,  7.7486e-04],
          [-1.2164e-01, -2.3468e-02, -2.1957e-02,  ...,  5.3902e-03,
            6.3896e-03, -3.7174e-03],
          ...,
          [-9.5367e-03, -1.9321e-03, -1.4229e-02,  ..., -1.4503e-02,
           -1.3895e-03,  1.0963e-02],
          [ 1.4544e-03, -5.3291e-03,  5.6076e-03,  ..., -2.0676e-03,
            2.1148e-04,  9.4299e-03],
          [-5.4283e-03,  6.3539e-05,  1.1017e-02,  ..., -2.2106e-03,
           -3.1769e-02,  3.9093e-02]], device='cuda:0', requires_grad=True),
  Parameter containing:
  tensor([[ 0.0065, -0.0371, -0.0054,  ..., -0.0308, -0.0266,  0.0204],
          [-0.0436,  0.0074,  0.0065,  ...,  0.0040,  0.0378,  0.0158],
          [ 0.0195,  0.0360, -0.0028,  ...,  0.0174,  0.0323, -0.0168],
          ...,
          [-0.0

In [9]:
print("lam sparse", parallel_tg_optimizer[0].param_groups[1]["lambda_sparse"])
print("rank", parallel_tg_optimizer[0].param_groups[1]["rank"])
print("sparse_ratio", parallel_tg_optimizer[0].param_groups[1]["sparse_ratio"])
print("update_proj_gap", parallel_tg_optimizer[0].param_groups[1]["update_proj_gap"])
print("update_proj_gap_end", parallel_tg_optimizer[0].param_groups[1]["update_proj_gap_end"])
print("update_proj_gap_mode", parallel_tg_optimizer[0].param_groups[1]["update_proj_gap_mode"])

lam sparse 0.05
rank 150
sparse_ratio 0.05
update_proj_gap 20
update_proj_gap_end 1000
update_proj_gap_mode linear


In [10]:
#that two goes to update gap scheduler
parallel_tg_optimizer[0].param_groups[1]['batch_size']
parallel_tg_optimizer[0].param_groups[1]['training_samples']

517

In [83]:
torchinfo.summary(model)

Layer (type:depth-idx)                        Param #
LlamaForCausalLM                              --
├─LlamaModel: 1-1                             --
│    └─Embedding: 2-1                         6,144,000
│    └─ModuleList: 2-2                        --
│    │    └─LlamaDecoderLayer: 3-1            700,800
│    └─LlamaRMSNorm: 2-3                      192
│    └─LlamaRotaryEmbedding: 2-4              --
├─Linear: 1-2                                 6,144,000
Total params: 12,988,992
Trainable params: 12,988,992
Non-trainable params: 0

In [11]:
list(map(lambda x: x[1].shape, list(model.named_parameters())))
model

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(32000, 192)
    (layers): ModuleList(
      (0): LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=192, out_features=192, bias=False)
          (k_proj): Linear(in_features=192, out_features=96, bias=False)
          (v_proj): Linear(in_features=192, out_features=96, bias=False)
          (o_proj): Linear(in_features=192, out_features=192, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=192, out_features=1024, bias=False)
          (up_proj): Linear(in_features=192, out_features=1024, bias=False)
          (down_proj): Linear(in_features=1024, out_features=192, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((192,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((192,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((192,), eps=1e-05)
    (rotary_emb): LlamaRotaryEm

In [12]:
from tdecomp.grad_proj.tensorgrad.prepared_tg import ParallelTG, ULTG

import importlib
from examples import experiment_utils
importlib.reload(experiment_utils)

def create_trainer(optimizer):
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset, #internally it creates Accelerator(torch.DataLoader(dataset)) that on gpu
        eval_dataset=eval_dataset,
        data_collator=data_collator,
        optimizers=optimizer,
        callbacks=[experiment_utils.UpdateGapMLflowCallback(), 
                experiment_utils.SystemMetricsCallback(),
                ]

    )
    trainer.add_callback(experiment_utils.PerplexityCallback(trainer.get_eval_dataloader(), max_batches=5, eval_every_n_steps=5))
    return trainer

trainer = create_trainer(parallel_tg_optimizer)
trainer.compute_loss_func
trainer.compute_loss

print("APROX BATCH SIZE in MB:", get_batch_size_mb(next(iter(trainer.get_train_dataloader()))))
print("Aprox model size MB:", get_model_size_mb(model))

APROX BATCH SIZE in MB: 0.046875
Aprox model size MB: 49.549072265625


In [13]:
print("Starting training...")
want_one_experiment = False
if (want_one_experiment):
    with mlflow.start_run(run_name="parallel_tg_rank_150_sparce_0.05"):
        trainer.train()
    torch.cuda.empty_cache()

Starting training...


In [14]:
import torch
print(torch.cuda.empty_cache())
print('alloc', torch.cuda.memory_allocated(0) / 1024 / 1024)
print('reserv', torch.cuda.memory_reserved(0) / 1024 / 1024)
print('max reserv', torch.cuda.max_memory_reserved(0) / 1024 / 1024)
print(torch.cuda.memory_stats(0))
print(torch.cuda.memory_snapshot())

None
alloc 49.55078125
reserv 54.0
max reserv 54.0
OrderedDict({'active.all.allocated': 264, 'active.all.current': 14, 'active.all.freed': 250, 'active.all.peak': 21, 'active.large_pool.allocated': 3, 'active.large_pool.current': 2, 'active.large_pool.freed': 1, 'active.large_pool.peak': 2, 'active.small_pool.allocated': 261, 'active.small_pool.current': 12, 'active.small_pool.freed': 249, 'active.small_pool.peak': 19, 'active_bytes.all.allocated': 104566272, 'active_bytes.all.current': 51957760, 'active_bytes.all.freed': 52608512, 'active_bytes.all.peak': 52428800, 'active_bytes.large_pool.allocated': 101580800, 'active_bytes.large_pool.current': 49152000, 'active_bytes.large_pool.freed': 52428800, 'active_bytes.large_pool.peak': 52428800, 'active_bytes.small_pool.allocated': 2985472, 'active_bytes.small_pool.current': 2805760, 'active_bytes.small_pool.freed': 179712, 'active_bytes.small_pool.peak': 2854912, 'allocated_bytes.all.allocated': 104566272, 'allocated_bytes.all.current': 51

In [16]:
if (False):
    RANKS = list(range(8, 192 + 31, 32))
    print("ranks", RANKS)
    SPARSE_RATIOS = [0.05, 0.1]
    LAMBDA_SPARCE = [0.05, 0.1, 0.5, 1]
    UPDATE_GAP = [10, 20, 40, 60, 100]
    for v in itertools.product(RANKS, SPARSE_RATIOS, LAMBDA_SPARCE, UPDATE_GAP):
        optimizer = create_optimizer(v[0], v[1], v[2], v[3])
        trainer = create_trainer(optimizer)
        with mlflow.start_run(run_name=f"parallel_tg_tiny_llm_{v}"):
            mlflow.log_params({
                "rank": v[0],
                "sparse_ratio": v[1],
                "lambda_sparse": v[2],
                "update_proj_gap": v[3]
            })
            trainer.train()
        torch.cuda.empty_cache()

In [17]:
print("hello")

hello


In [18]:
for batch in itertools.islice(dataloader, 3):
    print(dict(**{k: v for k, v in batch.items()}))
    break

{'input_ids': tensor([[    1, 29871,   450,  ...,     2,     2,     2],
        [    1, 29871, 17390,  ...,     2,     2,     2],
        [    1, 29871,   315,  ...,     2,     2,     2],
        [    1, 29871,   450,  ...,     2,     2,     2]], device='cuda:0'), 'attention_mask': tensor([[1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0]], device='cuda:0'), 'labels': tensor([[    1, 29871,   450,  ...,  -100,  -100,  -100],
        [    1, 29871, 17390,  ...,  -100,  -100,  -100],
        [    1, 29871,   315,  ...,  -100,  -100,  -100],
        [    1, 29871,   450,  ...,  -100,  -100,  -100]], device='cuda:0')}


In [19]:
inputs

{'input_ids': tensor([[    1,   512, 12321,  4006,  1919,   282,  3149,   358,   472, 19035,
          5845,   869,   450, 12321,  4006, 24909, 12919,  1045, 29877,   287,
          1075,   304,  1510,  1009, 16317, 27685,  2467,   411,   278, 11302,
           869,   940, 15569, 29871, 29896, 29953,  3291,   297, 29871, 29896,
         29941,  4943,  4259,  8090,   304,  8341,   278,  4259,   411, 29871,
         29896, 29896, 29900,  3291, 12420,  1546,   278,   349, 19636,  1144,
           322,   806,   284,   414,  1919,   322,   471,   278,  3815,   525,
         29879,  1900,  4847,   297,  1009,   937,  4513,  6410,   304,   278,
         12115,  8135,  1144,   297,   278, 29871, 29896, 29929, 29929, 29896,
         21631,  6536,  7412, 22450,   869,   940, 12919,  9259,   385,  2437,
          7018,   304,  5988,   278, 11443,  3815,   472,   278, 29871, 29896,
         29929, 29929, 29896,  7400,  6536,  1919]], device='cuda:0'), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1

In [20]:
next(iter(trainer.get_eval_dataloader()))

{'input_ids': tensor([[    1, 29871,   450,  ...,     2,     2,     2],
        [    1, 29871, 17390,  ...,     2,     2,     2],
        [    1, 29871,   315,  ...,     2,     2,     2],
        [    1, 29871,   450,  ...,     2,     2,     2]], device='cuda:0'), 'attention_mask': tensor([[1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0]], device='cuda:0'), 'labels': tensor([[    1, 29871,   450,  ...,  -100,  -100,  -100],
        [    1, 29871, 17390,  ...,  -100,  -100,  -100],
        [    1, 29871,   315,  ...,  -100,  -100,  -100],
        [    1, 29871,   450,  ...,  -100,  -100,  -100]], device='cuda:0')}

In [21]:
next(iter(dataloader))

{'input_ids': tensor([[    1, 29871,   450,  ...,     2,     2,     2],
        [    1, 29871, 17390,  ...,     2,     2,     2],
        [    1, 29871,   315,  ...,     2,     2,     2],
        [    1, 29871,   450,  ...,     2,     2,     2]], device='cuda:0'), 'attention_mask': tensor([[1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0]], device='cuda:0'), 'labels': tensor([[    1, 29871,   450,  ...,  -100,  -100,  -100],
        [    1, 29871, 17390,  ...,  -100,  -100,  -100],
        [    1, 29871,   315,  ...,  -100,  -100,  -100],
        [    1, 29871,   450,  ...,  -100,  -100,  -100]], device='cuda:0')}

In [22]:
logits = model(input_ids=gen_ids, attention_mask=torch.tensor([[1, 1, 1, 1, 1,0]])).logits
logits.shape

RuntimeError: The expanded size of the tensor (116) must match the existing size (6) at non-singleton dimension 3.  Target sizes: [1, 2, 116, 116].  Tensor sizes: [1, 1, 116, 6]

In [ ]:
target_logprobs = torch.tensor([1, 2, 3])
shift_mask = torch.tensor([1, 1, 0]).bool()
target_logprobs.masked_select(shift_mask)
shift_mask.sum(dim=-1)

tensor(2)

In [ ]:
v = [3000000, 1, 1, 10000]

optimizer = create_optimizer(v[0], v[1], v[2], v[3])
trainer = create_trainer(optimizer)
# mlflow.config.enable_system_metrics_logging()
# mlflow.config.set_system_metrics_sampling_interval(1)
with mlflow.start_run(run_name=f"parallel_tg_tiny_llm_{v}_another_sched",):
    mlflow.log_params({
        "rank": v[0],
        "sparse_ratio": v[1],
        "lambda_sparse": v[2],
        "update_proj_gap": v[3] #если fixed режим - тогда обновляется с этого шага и на эту величину. У других mode не совпадает.
    })
    trainer.train()
torch.cuda.empty_cache()

2026/02/26 09:41:31 WARNING mlflow.utils.git_utils: Failed to import Git (the Git executable is probably not on your PATH), so Git SHA is not available. Error: Failed to initialize: Bad git executable.
The git executable must be specified in one of the following ways:
    - be included in your $PATH
    - be set via $GIT_PYTHON_GIT_EXECUTABLE
    - explicitly set via git.refresh(<full-path-to-git-executable>)

All git commands will error until this is rectified.

This initial message can be silenced or aggravated in the future by setting the
$GIT_PYTHON_REFRESH environment variable. Use one of the following values:
    - quiet|q|silence|s|silent|none|n|0: for no message or exception
    - warn|w|warning|log|l|1: for a warning message (logging level CRITICAL, displayed by default)
    - error|e|exception|raise|r|2: for a raised exception

Example:
    export GIT_PYTHON_REFRESH=quiet

2026/02/26 09:41:32 INFO mlflow.system_metrics.system_metrics_monitor: Started monitoring system metrics

### Using Composite Projector Configuration ###
    => Swapping projectors to ensure smaller one is first
    => Sizes after swap: first=0.25, second=3000000.0
Update gap scheduler: <tdecomp.grad_proj.tensorgrad.projectors.update_gap_scheduler.UpdateGapScheduler object at 0x7e472f0dc770>
UnstructuredSparseProjector initialized with sparse_ratio=0.25, sparse_type=topk, scale_by_mask_ratio=False
    => First projector: unstructured_sparse
    => Second projector: low_rank
### Using Composite Projector Configuration ###
    => Swapping projectors to ensure smaller one is first
    => Sizes after swap: first=0.25, second=3000000.0
Update gap scheduler: <tdecomp.grad_proj.tensorgrad.projectors.update_gap_scheduler.UpdateGapScheduler object at 0x7e472f373e60>
UnstructuredSparseProjector initialized with sparse_ratio=0.25, sparse_type=topk, scale_by_mask_ratio=False
    => First projector: unstructured_sparse
    => Second projector: low_rank
### Using Composite Projector Configuration ###
  

Step,Training Loss
5,5.117416
10,4.792957
15,4.032818
20,4.450734
25,4.280299
30,4.113808
35,4.477179
40,4.338981


2026/02/26 09:41:42 INFO mlflow.system_metrics.system_metrics_monitor: Stopping system metrics monitoring...
2026/02/26 09:41:42 INFO mlflow.system_metrics.system_metrics_monitor: Successfully terminated system metrics monitoring!


🏃 View run parallel_tg_tiny_llm_[3000000, 1, 1, 10000]_another_sched at: http://172.19.0.1:5000/#/experiments/58/runs/392f3e5ff51141028d745a900de6ecfa
🧪 View experiment at: http://172.19.0.1:5000/#/experiments/58


KeyboardInterrupt: 

In [1]:
import mlflow.experiments


mlflow.experiments

/tdecomp/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


<module 'mlflow.experiments' from '/tdecomp/.venv/lib/python3.12/site-packages/mlflow/experiments.py'>

In [9]:
ranked_models = mlflow.search_logged_models(
    # experiment_ids=["58"]
)

In [10]:
ranked_models

,artifact_location,creation_timestamp,experiment_id,last_updated_timestamp,metrics,model_id,model_type,name,params,source_run_id,status,status_message,tags
0,/tdecomp/examples/mlruns/0/models/m-4b20edc7a9...,1771326255521,0,1771326266005,None,m-4b20edc7a9ae4688afc71e4f57fbce5a,None,pytorch_model_example_glazkov,{},87534584b37d4174a4ce0ee66cd32544,READY,None,"{'mlflow.source.name': 'tensorgrad_run.ipynb',..."


In [101]:
run_example = mlFlowClient.get_run("cd79c087849b4d53b0c659f9f044781c")
run_example.data.metrics.keys()

dict_keys(['gpu_memory_used_mb', 'gpu_memory_reserved_mb', 'cpu_percent', 'ug_next_update', 'ug_next_update_changed', 'perplexity', 'loss', 'grad_norm', 'learning_rate', 'epoch', 'train_runtime', 'train_samples_per_second', 'train_steps_per_second', 'total_flos', 'train_loss'])

In [105]:
mlFlowClient.get_run("cd79c087849b4d53b0c659f9f044781c").data.metrics

{'gpu_memory_used_mb': 18043.552256,
 'gpu_memory_reserved_mb': 18238.930944,
 'cpu_percent': 2.7,
 'ug_next_update': 10001.0,
 'ug_next_update_changed': 0.0,
 'perplexity': 72.23211669921875,
 'loss': 3.434459686279297,
 'grad_norm': 40.87346267700195,
 'learning_rate': 1.000000000000001e-16,
 'epoch': 3.0,
 'train_runtime': 93.0974,
 'train_samples_per_second': 16.66,
 'train_steps_per_second': 4.189,
 'total_flos': 32614141722624.0,
 'train_loss': 4.06822821299235}

In [96]:
run_example = mlFlowClient.get_run("cd79c087849b4d53b0c659f9f044781c")
run_example.data.metrics.keys()
runs = mlFlowClient.search_runs(["58"])


In [99]:
runs[1].info.run_id

'8e23e58dd90f47b697c9e92ae0d8f4bd'

In [51]:
MLFLOW_KEYS = run_example.data.metrics.keys()

hist = mlFlowClient.get_metric_history("cd79c087849b4d53b0c659f9f044781c", "cpu_percent")
hist

[<Metric: dataset_digest=None, dataset_name=None, key='cpu_percent', model_id=None, run_id=None, step=1, timestamp=1772032078886, value=3.1>,
 <Metric: dataset_digest=None, dataset_name=None, key='cpu_percent', model_id=None, run_id=None, step=2, timestamp=1772032079414, value=3.5>,
 <Metric: dataset_digest=None, dataset_name=None, key='cpu_percent', model_id=None, run_id=None, step=3, timestamp=1772032079647, value=4.6>,
 <Metric: dataset_digest=None, dataset_name=None, key='cpu_percent', model_id=None, run_id=None, step=4, timestamp=1772032079888, value=3.4>,
 <Metric: dataset_digest=None, dataset_name=None, key='cpu_percent', model_id=None, run_id=None, step=5, timestamp=1772032080132, value=4.0>,
 <Metric: dataset_digest=None, dataset_name=None, key='cpu_percent', model_id=None, run_id=None, step=6, timestamp=1772032080493, value=3.0>,
 <Metric: dataset_digest=None, dataset_name=None, key='cpu_percent', model_id=None, run_id=None, step=7, timestamp=1772032080737, value=4.3>,
 <Metr

In [61]:
list(MLFLOW_KEYS)

['gpu_memory_used_mb',
 'gpu_memory_reserved_mb',
 'cpu_percent',
 'ug_next_update',
 'ug_next_update_changed',
 'perplexity',
 'loss',
 'grad_norm',
 'learning_rate',
 'epoch',
 'train_runtime',
 'train_samples_per_second',
 'train_steps_per_second',
 'total_flos',
 'train_loss']

In [55]:
import pandas as pd


df = pd.DataFrame([{"value":m.value, "timestamp":m.timestamp, "step":m.} for m in hist])
df
df
hist

SyntaxError: invalid syntax (822219505.py, line 4)

,step,gpu_memory_used_mb,gpu_memory_reserved_mb,cpu_percent,ug_next_update,ug_next_update_changed,perplexity,loss,grad_norm,learning_rate,epoch,train_runtime,train_samples_per_second,train_steps_per_second,total_flos,train_loss
0,5,18043.589120,18211.667968,4.0,10001.0,0.0,134.736221,5.117416,17.261621,1.000000e-04,0.038462,NaN,NaN,NaN,NaN,NaN
1,10,18043.589120,18211.667968,3.6,10001.0,0.0,112.065697,4.792957,8.869555,1.000000e-04,0.076923,NaN,NaN,NaN,NaN,NaN
2,15,18043.589120,18211.667968,4.5,10001.0,0.0,99.087708,4.032818,12.194827,1.000000e-04,0.115385,NaN,NaN,NaN,NaN,NaN
3,20,18043.589120,18211.667968,2.9,10001.0,0.0,89.679573,4.450734,14.005716,1.000000e-04,0.153846,NaN,NaN,NaN,NaN,NaN
4,25,18043.589120,18211.667968,3.8,10001.0,0.0,82.112648,4.280299,12.884663,1.000000e-04,0.192308,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
74,375,18043.589120,18236.833792,4.6,10001.0,0.0,72.232117,4.128750,7.118540,1.000000e-16,2.884615,NaN,NaN,NaN,NaN,NaN
75,380,18043.589120,18236.833792,3.3,10001.0,0.0,72.232117,4.210194,9.383719,1.000000e-16,2.923077,NaN,NaN,NaN,NaN,NaN
76,385,18043.589120,18236.833792,6.2,10001.0,0.0,72.232117,3.807521,10.515495,1.000000e-16,2.961538,NaN,NaN,NaN,NaN,NaN
77,390,18043.552256,18238.930944,2.7,10001.0,0.0,72.232117,3.434460,40.873463,1.000000e-16,3.000000,93.0974,16.66,4.189,3.261414e+13,4.068228


In [100]:
df["run_id"] = 5
df

,step,gpu_memory_used_mb,gpu_memory_reserved_mb,cpu_percent,ug_next_update,ug_next_update_changed,perplexity,loss,grad_norm,learning_rate,epoch,train_runtime,train_samples_per_second,train_steps_per_second,total_flos,train_loss,run_id
0,5,18043.589120,18211.667968,4.0,10001.0,0.0,134.736221,5.117416,17.261621,1.000000e-04,0.038462,NaN,NaN,NaN,NaN,NaN,5
1,10,18043.589120,18211.667968,3.6,10001.0,0.0,112.065697,4.792957,8.869555,1.000000e-04,0.076923,NaN,NaN,NaN,NaN,NaN,5
2,15,18043.589120,18211.667968,4.5,10001.0,0.0,99.087708,4.032818,12.194827,1.000000e-04,0.115385,NaN,NaN,NaN,NaN,NaN,5
3,20,18043.589120,18211.667968,2.9,10001.0,0.0,89.679573,4.450734,14.005716,1.000000e-04,0.153846,NaN,NaN,NaN,NaN,NaN,5
4,25,18043.589120,18211.667968,3.8,10001.0,0.0,82.112648,4.280299,12.884663,1.000000e-04,0.192308,NaN,NaN,NaN,NaN,NaN,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
74,375,18043.589120,18236.833792,4.6,10001.0,0.0,72.232117,4.128750,7.118540,1.000000e-16,2.884615,NaN,NaN,NaN,NaN,NaN,5
75,380,18043.589120,18236.833792,3.3,10001.0,0.0,72.232117,4.210194,9.383719,1.000000e-16,2.923077,NaN,NaN,NaN,NaN,NaN,5
76,385,18043.589120,18236.833792,6.2,10001.0,0.0,72.232117,3.807521,10.515495,1.000000e-16,2.961538,NaN,NaN,NaN,NaN,NaN,5
77,390,18043.552256,18238.930944,2.7,10001.0,0.0,72.232117,3.434460,40.873463,1.000000e-16,3.000000,93.0974,16.66,4.189,3.261414e+13,4.068228,5
